# 🌲 Cypress: GPU-Accelerated PCB Placement on Google Colab

This notebook runs the prebuilt **Cypress** Apptainer container (`cypress.sif`) on Google Colab's Free Tier NVIDIA T4 GPU.

### How It Works (Protocol Option A: Google Drive Batch Queue)
1. Mounts Google Drive (`/content/drive/MyDrive/Cypress_Colab`).
2. Installs Apptainer in seconds (unprivileged, no docker daemon required).
3. Passes the Colab T4 GPU to the container via `--nv`.
4. Starts the **Queue Worker**, which polls `queue/` on Google Drive, runs placement, and outputs final `.gp.pl` coordinates and plots to `completed/`.

## Step 1: Check GPU & Install Apptainer

In [ ]:
# Verify GPU allocation (Google Colab Free Tier typically gives an NVIDIA Tesla T4)
!nvidia-smi

# Install Apptainer via official PPA (runs in ~10 seconds)
!sudo apt-get update && sudo apt-get install -y software-properties-common
!sudo add-apt-repository -y ppa:apptainer/ppa
!sudo apt-get update -qq
!sudo apt-get install -y -qq apptainer

# Configure Apptainer for Colab container permissions
!sudo sed -i 's/^root default capabilities = full/root default capabilities = no/' /etc/apptainer/apptainer.conf
!sudo sed -i 's/^enable overlay = .*/enable overlay = driver/' /etc/apptainer/apptainer.conf
!sudo sed -i 's/^enable underlay = .*/enable underlay = preferred/' /etc/apptainer/apptainer.conf
!sudo sed -i 's/^max underlay binds = .*/max underlay binds = 5000/' /etc/apptainer/apptainer.conf

# Register Colab's NVIDIA driver libraries in ldconfig so Apptainer finds libcuda.so.1
!echo "/usr/lib64-nvidia" | sudo tee /etc/ld.so.conf.d/nvidia.conf
!sudo ldconfig

# Initialize host CUDA to ensure /dev/nvidia-uvm is instantiated
import torch
print("Host CUDA available:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
if torch.cuda.is_available():
    torch.zeros(1).cuda()

!apptainer version

## Step 2: Mount Google Drive & Define Paths

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount your Google Drive
drive.mount('/content/drive')

# Set the path to your shared workspace folder in Google Drive
WORKSPACE = Path('/content/drive/MyDrive/Cypress_Colab')
SIF_PATH = WORKSPACE / 'cypress.sif'

# Ensure required queue directories exist
for sub in ['queue', 'running', 'completed', 'failed']:
    (WORKSPACE / sub).mkdir(parents=True, exist_ok=True)

print(f"Workspace: {WORKSPACE}")
print(f"SIF Image Exists: {SIF_PATH.exists()}")
if not SIF_PATH.exists():
    print("\n[!] WARNING: cypress.sif not found! Please upload cypress.sif to Google Drive under 'Cypress_Colab/'.")

## Step 3: Smoke Test GPU Inside the Container

In [ ]:
# Verify that PyTorch inside the prebuilt container sees Colab's GPU
!apptainer exec --nv --no-privs \
  -B /usr/lib64-nvidia:/usr/lib64-nvidia \
  --env LD_LIBRARY_PATH="/usr/lib64-nvidia:/usr/local/nvidia/lib64" \
  "{SIF_PATH}" python3 -c \
  "import torch, dreamplace.ops.place_io.place_io as pio; print('PyTorch Version:', torch.__version__); print('CUDA Available:', torch.cuda.is_available()); print('Device Name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')"

## Step 4: Run the Queue Worker (Batch Mode)

Run this cell while you want Colab to process jobs. Whenever you drop a job into Google Drive's `queue/` folder, this cell will process it and move the result to `completed/`.

In [ ]:
# Start queue worker daemon (uses updated script from Google Drive if available)
import os
worker_py = "/workspace/colab_queue_worker.py" if os.path.exists(f"{WORKSPACE}/colab_queue_worker.py") else "/Cypress/scripts/colab_queue_worker.py"

!apptainer exec --nv --no-privs \
  -B /usr/lib64-nvidia:/usr/lib64-nvidia \
  -B "{WORKSPACE}:/workspace" \
  --env LD_LIBRARY_PATH="/usr/lib64-nvidia:/usr/local/nvidia/lib64" \
  "{SIF_PATH}" \
  python3 {worker_py} --workspace /workspace --poll-interval 5

## (Optional) Step 5: Direct Single-Job Execution

If you prefer running a single benchmark interactively without the queue, run this cell:

In [ ]:
# Example: run the built-in benchmark directly
!apptainer exec --nv --no-privs \
  -B /usr/lib64-nvidia:/usr/lib64-nvidia \
  --env LD_LIBRARY_PATH="/usr/lib64-nvidia:/usr/local/nvidia/lib64" \
  "{SIF_PATH}" python3 /Cypress/dreamplace/Placer.py /Cypress/configs/test_small1.json